In [46]:
import re

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np


In [52]:
df_in = pd.read_excel(
    "data/DI_Card_Inventory_v2.4.xlsx",
    sheet_name="Deck",
    skiprows=1,
    usecols="B:C,D:R",
    nrows=44,
)
df_in = df_in.drop(columns=["Type"])

df_in.columns = df_in.columns.str.replace("\n", " ", regex=False)
df_in = df_in.replace("\n", " ", regex=True)

cols_to_convert = [
    "Green Space",
    "Blue Space",
    "Yellow Space",
    "Emperor Space",
    "Guild Space",
    "Bene Space",
    "Fremen Space",
]
for col in cols_to_convert:
    df_in[col] = df_in[col].eq("X")

cols_to_convert_int = ["Reveal persuasion", "Reveal Swords"]
for col in cols_to_convert_int:
    df_in[col] = pd.to_numeric(df_in[col].replace("-", 0), errors="coerce").fillna(0).astype(int)

# df_in["Faction"] = df_in["Faction"].replace("-", np.nan)
# df_in = pd.get_dummies(df_in, columns=["Faction"], dummy_na=False)


df_in.info()
df_in.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44 entries, 0 to 43
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Card Name          44 non-null     object
 1   Count              44 non-null     int64 
 2   Cost               44 non-null     int64 
 3   Various            44 non-null     object
 4   Agent Ability      44 non-null     object
 5   Reveal persuasion  44 non-null     int64 
 6   Reveal Swords      44 non-null     int64 
 7   Reveal Ability     43 non-null     object
 8   Green Space        44 non-null     bool  
 9   Blue Space         44 non-null     bool  
 10  Yellow Space       44 non-null     bool  
 11  Emperor Space      44 non-null     bool  
 12  Guild Space        44 non-null     bool  
 13  Bene Space         44 non-null     bool  
 14  Fremen Space       44 non-null     bool  
 15  Faction            44 non-null     object
dtypes: bool(7), int64(4), object(5)
memory usage: 

/var/folders/q6/pntlsv6s1j19xm8fhz9g32fr0000gn/T/ipykernel_85862/234805404.py:23: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_in[col] = df_in[col].replace({"X": 1, "-": 0}).astype(bool)
/var/folders/q6/pntlsv6s1j19xm8fhz9g32fr0000gn/T/ipykernel_85862/234805404.py:23: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_in[col] = df_in[col].replace({"X": 1, "-": 0}).astype(bool)
/var/folders/q6/pntlsv6s1j19xm8fhz9g32fr0000gn/T/ipykernel_85862/234805404.py:23: FutureWarning: Downcasting behavior in `replace` is deprecated and will 

,Card Name,Count,Cost,Various,Agent Ability,Reveal persuasion,Reveal Swords,Reveal Ability,Green Space,Blue Space,Yellow Space,Emperor Space,Guild Space,Bene Space,Fremen Space,Faction
0,Arrakis Recruiter,2,2,-,+1 Troop,1,1,-,False,True,False,False,False,False,False,-
1,Assassination Mission,2,1,-,When trashed by another card or effect: +4 Solari,0,1,+1 Solari,False,False,False,False,False,False,False,-
2,Bene Gesserit Initiate,2,3,-,Draw 1 card,1,0,-,True,True,True,False,False,False,False,Bene Gesserit
3,Bene Gesserit Sister,3,3,-,-,0,0,+2 persuasion Or +2 Swords,True,False,False,False,False,True,False,Bene Gesserit
4,Carryall,1,5,-,Double base spice harvest (not bonus),1,0,+1 Spice,False,False,True,False,False,False,False,-


In [ ]:
space_cols = [
    "Green Space",
    "Blue Space",
    "Yellow Space",
    "Emperor Space",
    "Guild Space",
    "Bene Space",
    "Fremen Space",
]

df_in["n_spaces"] = df_in[space_cols].sum(axis=1).astype(int)
df_in["reveal_total_combat"] = (
    df_in["Reveal persuasion"] + df_in["Reveal Swords"]
).astype(int)

various_str = df_in["Various"].astype(str).str.strip()
df_in["has_with_buy"] = various_str.str.startswith("With Buy:", na=False).astype(int)


def _ability_blob(row: pd.Series) -> str:
    parts = [
        str(row["Agent Ability"]),
        str(row["Reveal Ability"]) if pd.notna(row["Reveal Ability"]) else "",
        str(row["Various"]),
    ]
    return "\n".join(parts)


blob = df_in.apply(_ability_blob, axis=1)
blob_lower = blob.str.lower()

_signed = re.compile(r"([+-])\s*(\d+)")
df_in["text_signed_bonus_sum"] = blob.apply(
    lambda s: sum(
        (1 if m.group(1) == "+" else -1) * int(m.group(2))
        for m in _signed.finditer(s)
    )
).astype(int)

_pay = re.compile(r"(?i)pay\s+(\d+)")
df_in["text_n_pay"] = blob.apply(lambda s: len(_pay.findall(s))).astype(int)

# " Or " between modes; also "draw ... or draw"
df_in["text_or_count"] = blob_lower.str.count(r"\s or \s")

_scale = re.compile(
    r"(?i)(for each|when trashed|when .{0,40}:|if you have|alliance|\bbond\b"
    r"|with another|with \d+ \w+ |each opponent|costs\s+\d+\s+less|including this one)"
)
df_in["text_scales_or_conditional"] = blob.apply(lambda s: bool(_scale.search(s))).astype(
    int
)

fac = df_in["Faction"].astype(str)
df_in["is_neutral_faction"] = (fac == "-").astype(int)
df_in["faction_emperor"] = fac.str.contains("Emperor", regex=False).astype(int)
df_in["faction_bene_gesserit"] = fac.str.contains("Bene Gesserit", regex=False).astype(
    int
)
df_in["faction_fremen"] = fac.str.contains("Fremen", regex=False).astype(int)
df_in["faction_spacing_guild"] = fac.str.contains("Spacing Guild", regex=False).astype(
    int
)

FEATURE_COLS = [
    "Count",
    "Reveal persuasion",
    "Reveal Swords",
    "n_spaces",
    "reveal_total_combat",
    "has_with_buy",
    "text_signed_bonus_sum",
    "text_n_pay",
    "text_or_count",
    "text_scales_or_conditional",
    "is_neutral_faction",
    "faction_emperor",
    "faction_bene_gesserit",
    "faction_fremen",
    "faction_spacing_guild",
    "Green Space",
    "Blue Space",
    "Yellow Space",
    "Emperor Space",
    "Guild Space",
    "Bene Space",
    "Fremen Space",
]


In [61]:
X = df_in[FEATURE_COLS]
y = df_in["Cost"]

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import LeaveOneOut

loo = LeaveOneOut()
rf_params = {"n_estimators": 1024, "random_state": 42, "ccp_alpha": 0.06}

y_pred_loo = np.empty(len(y), dtype=float)
for train_idx, test_idx in loo.split(X):
    est = RandomForestRegressor(**rf_params)
    est.fit(X.iloc[train_idx], y.iloc[train_idx])
    y_pred_loo[test_idx] = est.predict(X.iloc[test_idx])

mae_loo = mean_absolute_error(y, y_pred_loo)
rmse_loo = root_mean_squared_error(y, y_pred_loo)
mae_mean_baseline = mean_absolute_error(y, np.full(len(y), y.mean()))

print(f"LOO MAE (RandomForest): {mae_loo:.3f}")
print(f"LOO RMSE (RandomForest): {rmse_loo:.3f}")
print(f"MAE (predict mean cost={y.mean():.3f}): {mae_mean_baseline:.3f}")

In [ ]:
from sklearn.linear_model import ElasticNetCV
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import LeaveOneOut
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

loo = LeaveOneOut()
enet_loo_preds = np.empty(len(y), dtype=float)
for train_idx, test_idx in loo.split(X):
    enet = Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "enet",
                ElasticNetCV(
                    l1_ratio=[0.25, 0.5, 0.75, 0.9, 0.99],
                    cv=5,
                    random_state=42,
                    max_iter=25_000,
                ),
            ),
        ]
    )
    enet.fit(X.iloc[train_idx], y.iloc[train_idx])
    enet_loo_preds[test_idx] = enet.predict(X.iloc[test_idx])

mae_enet_loo = mean_absolute_error(y, enet_loo_preds)
rmse_enet_loo = root_mean_squared_error(y, enet_loo_preds)
print(f"LOO MAE (Elastic Net): {mae_enet_loo:.3f}")
print(f"LOO RMSE (Elastic Net): {rmse_enet_loo:.3f}")

enet_full = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "enet",
            ElasticNetCV(
                l1_ratio=[0.25, 0.5, 0.75, 0.9, 0.99],
                cv=5,
                random_state=42,
                max_iter=25_000,
            ),
        ),
    ]
)
enet_full.fit(X, y)
en = enet_full.named_steps["enet"]
print(f"Full-data CV: alpha={en.alpha_:.4f}, l1_ratio={en.l1_ratio_}")

coef = pd.Series(en.coef_, index=FEATURE_COLS).sort_values(key=abs, ascending=False)
print("Coefficients (standardized features):")
print(coef.to_string())


In [ ]:
coef_plot = coef.reindex(coef.abs().sort_values().index)
colors = np.where(coef_plot.values >= 0, "#3182bd", "#e6550d")

fig_h = max(4.5, 0.38 * len(coef_plot))
fig, ax = plt.subplots(figsize=(9, fig_h))
y_pos = np.arange(len(coef_plot))
ax.barh(y_pos, coef_plot.values, color=colors, edgecolor="white", linewidth=0.5)
ax.axvline(0, color="#333333", linewidth=0.9)
ax.set_yticks(y_pos)
ax.set_yticklabels(coef_plot.index)
ax.set_xlabel("Coefficient (one SD change in scaled feature → change in cost)")
ax.set_title("Elastic Net: coefficient size by feature")

xlim = ax.get_xlim()
pad = (xlim[1] - xlim[0]) * 0.02
for i, v in enumerate(coef_plot.values):
    if v >= 0:
        ha, x = "left", v + pad
    else:
        ha, x = "right", v - pad
    ax.text(x, i, f"{v:+.3f}", va="center", ha=ha, fontsize=9, color="#333333")

ax.invert_yaxis()
plt.tight_layout()
plt.show()


In [76]:
# Full-data fit for residual inspection (same hyperparameters as LOO cell)
model = RandomForestRegressor(n_estimators=1024, random_state=42, ccp_alpha=0.06)

model.fit(X, y)
df_in.assign(
    pred=model.predict(X), diff=lambda df: df["Cost"] - df["pred"]
).sort_values("diff", key=abs, ascending=False)


,Card Name,Count,Cost,Various,Agent Ability,Reveal persuasion,Reveal Swords,Reveal Ability,Green Space,Blue Space,Yellow Space,Emperor Space,Guild Space,Bene Space,Fremen Space,Faction,pred,diff
8,Dr. Yueh,1,1,-,Draw 1 card,1,0,NaN,False,True,False,False,False,False,False,-,2.610907,-1.610907
6,Choam Directorship,1,8,With Buy: +1 Influence to all 4 Factions,-,0,0,+3 Solari,False,False,False,False,False,False,False,-,6.398113,1.601887
20,Kwisatz Haderach,1,8,-,Send one of your agents from anywhere to any b...,0,0,-,True,True,True,True,True,True,True,Bene Gesserit,6.419091,1.580909
42,Worm Riders,2,6,-,+2 Spice,0,0,Having 2 Fremen Influence: +4 Swords and Havin...,False,True,True,False,False,False,False,Fremen,4.641193,1.358807
1,Assassination Mission,2,1,-,When trashed by another card or effect: +4 Solari,0,1,+1 Solari,False,False,False,False,False,False,False,-,2.336898,-1.336898
39,Test of Humanity,1,3,-,Opponents discard 1 card or lose 1 deployed Troop,2,0,-,True,True,False,False,False,True,False,Bene Gesserit,4.285048,-1.285048
16,Guild Bankers,1,3,-,-,0,0,The Spice Must Flow costs 3 less this turn,True,False,False,True,True,False,False,Spacing Guild,4.266570,-1.266570
18,Gurney Halleck,1,6,-,+2 Troops and Draw 1 card,2,0,Pay 3 solari: +2 Troops to Garrison or Conflict,False,True,False,False,False,False,False,-,4.742041,1.257959
7,Crysknife,1,3,-,+1 Solari,0,1,Fremen Bond: +1 Influence with Fremen,False,False,True,False,False,False,True,Fremen,4.116687,-1.116687
34,Smuggler's Thopter,2,4,-,With 2 Spacing Guild Influence: Draw 2 cards,1,0,+1 Spice,False,False,True,False,False,False,False,Spacing Guild,2.899830,1.100170
